In [21]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [22]:
# load data
df = pd.read_csv("data/preprocessing_data.csv", keep_default_na=False)

# initial split train and test set based on within mandate period
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=20240417, 
    stratify=df["within_mandate_period"]
)

In [23]:
# define feature elimination rules and target variables for each model
# 1: face mask behaviour, 2: protective behaviour
# a: before mandate, b: after mandate
model_configs = {
    "model_1": {
        "target": "face_mask_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "endtime"]
    },
    "model_1a": {
        "target": "face_mask_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "endtime", "within_mandate_period"]
    },
    "model_1b": {
        "target": "face_mask_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "endtime", "within_mandate_period"]
    },
    "model_2": {
        "target": "protective_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "protective_behaviour_nomask_scale", "endtime"]
    },
    "model_2a": {
        "target": "protective_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "protective_behaviour_nomask_scale", "endtime", "within_mandate_period"]
    },
    "model_2b": {
        "target": "protective_behaviour_binary",
        "drop_cols": ["RecordNo", "face_mask_behaviour_scale", "protective_behaviour_scale", 
                      "face_mask_behaviour_binary", "protective_behaviour_binary", "protective_behaviour_nomask_scale", "endtime", "within_mandate_period"]
    }
}


In [24]:
# define conditions for each model
def get_filter_condition(model_name, df_subset):
    if model_name in ["model_1", "model_2"]:
        return pd.Series(True, index=df_subset.index)
    elif model_name in ["model_1a", "model_2a"]:
        # before mandate
        return (df_subset["endtime"] < "2022-01-01") & (df_subset["within_mandate_period"] == 0)
    elif model_name in ["model_1b", "model_2b"]:
        # after mandate
        return df_subset["within_mandate_period"] == 1
    return pd.Series(True, index=df_subset.index)


In [25]:
# process and save the datasets of all models
label_encoder = LabelEncoder()

for name, config in model_configs.items():
    # extract target and features
    target_col = config["target"]
    feature_cols = [c for c in df.columns if c not in config["drop_cols"]]
    
    # filter conditions
    train_mask = get_filter_condition(name, train_df)
    test_mask = get_filter_condition(name, test_df)
    
    # extract X and y
    X_train = train_df.loc[train_mask, feature_cols]
    X_test = test_df.loc[test_mask, feature_cols]
    
    # prevent data leakage
    y_train = label_encoder.fit_transform(train_df.loc[train_mask, target_col])
    y_test = label_encoder.fit_transform(test_df.loc[test_mask, target_col])
    
    # save all sets
    X_train.to_csv(f"data/X_train_{name}.csv", index=False)
    X_test.to_csv(f"data/X_test_{name}.csv", index=False)
    pd.DataFrame({'y_train': y_train}).to_csv(f"data/y_train_{name}.csv", index=False)
    pd.DataFrame({'y_test': y_test}).to_csv(f"data/y_test_{name}.csv", index=False)